# 🌌 OMEGA v5: Supreme Domination Engine
**Goal: 1000+ Elo | Competition: Orbit Wars — Kaggle Simulation**

OMEGA v5 is the absolute pinnacle of strategic AI for the Orbit Wars environment. It completely redefines the tactical metagame by introducing mechanics like **Simultaneous Counter-Rush**, **Planet Triage**, **Vulnerability Window Exploits**, and a highly sophisticated **Death Ball Endgame** logic.

## 👑 The 8-Layer Architecture
1. **Physics:** Orbital prediction, Solar Bypass routing, Speed curve exploitation.
2. **World Model:** Timeline simulation, Binary Search evaluation.
3. **Economic Mode:** 5-tier system (SNOWBALL to PANIC).
4. **Policy Builder:** Dynamic reserves, Death Ball awareness.
5. **Scoring:** 14 multipliers targeting vulnerability and pressure points.
6. **Mission Engine:** 12 execution types including Counter-Rush.
7. **Executor:** Hyper-Tsunami and Concentration of Force.
8. **Endgame:** Precise ship-delta tracking for 'Defend' vs 'All-in' transitions.

## 1. Physics: Solar Bypass (Reaching the Unreachable)
A critical flaw in most agents is giving up when the sun blocks the direct path to a target. OMEGA v5 calculates the exact tangent angles to bypass the sun's danger zone, allowing it to strike planets that enemies believe are safe.

In [ ]:
def bypass_angle(sx, sy, sr, tx, ty, tr, clockwise=True):
    """
    v5 NEW: Find tangent route around the sun danger zone.
    Computes a waypoint just outside the sun's destruction radius, then routes to target.
    """
    danger_r = SUN_R + SUN_SAFETY + 0.6
    cx, cy   = CENTER_X, CENTER_Y
    to_sun_d = dist(sx, sy, cx, cy)
    
    if to_sun_d <= danger_r: return None
    
    base_angle = math.atan2(cy - sy, cx - sx)
    half_ang   = math.asin(min(1.0, danger_r / to_sun_d))
    tang_angle = base_angle + (half_ang + 0.18 if clockwise else -(half_ang + 0.18))
    tang_dist  = math.sqrt(max(0, to_sun_d**2 - danger_r**2)) + 2.5
    
    # Calculate Waypoint
    wx = sx + math.cos(tang_angle) * tang_dist
    wy = sy + math.sin(tang_angle) * tang_dist
    
    # Validate and return
    direct_to_t = safe_angle_dist(wx, wy, 0.5, tx, ty, tr)
    if direct_to_t is None: return None
        
    launch_angle = math.atan2(wy - sy, wx - sx)
    total_d = tang_dist + dist(wx, wy, tx, ty)
    return launch_angle, total_d

## 2. Death Ball Endgame: Precision Win/Lose Logic
The most critical late-game insight: **If you're already winning, stop attacking.** In the final 60 turns, OMEGA tracks the exact ship delta. If our ratio is `≥ 1.08`, we enter `[DEFEND]` mode, halting all offensive operations and hoarding ships to guarantee a mathematical victory. If the ratio drops to `≤ 0.95`, it enters `[ALL-IN]`, abandoning defenses for a desperate final strike.

In [ ]:
DEATH_BALL_TURNS       = 60    # Activate in last 60 turns
DEATH_BALL_WIN_MARGIN  = 1.08  # if we're 8%+ ahead, STOP attacking, defend
DEATH_BALL_LOSE_MARGIN = 0.95  # if within 5%, go all-in immediately

def death_ball_status(world):
    """
    Decide whether to defend the lead or go all-in.
    Returns ('defend', margin), ('allin', deficit) or ('press', ratio)
    """
    if world.remaining > DEATH_BALL_TURNS: return None, 0
    
    my_t = world.my_total
    en_t = world.enemy_total
    
    if en_t == 0: return 'defend', my_t
    
    ratio = my_t / en_t
    
    if ratio >= DEATH_BALL_WIN_MARGIN: 
        return 'defend', ratio
    if ratio <= DEATH_BALL_LOSE_MARGIN: 
        return 'allin', en_t - my_t + 1
        
    return 'press', ratio  # keep attacking but also hold

## 3. Simultaneous Counter-Rush & Planet Triage
OMEGA v5 introduces two highly advanced strategic concepts:
1. **Simultaneous Counter-Rush:** If the enemy sends an early, massive rush fleet at us, OMEGA doesn't just defend passively. It identifies the enemy's highest-production "home" planet, applies a `2.00x` scoring multiplier, and attacks it simultaneously. "The best defense is destroying their economy."
2. **Planet Triage (Strategic Abandonment):** If the cost to defend a planet is mathematically greater than `3.0x` its remaining value, OMEGA will abandon it and redirect those defensive ships into an offensive strike.

In [ ]:
# --- EXCERPT: PLANET TRIAGE ---
def compute_triage_set(world, policy):
    abandon = set()
    my_sorted = sorted(world.my_planets, key=lambda p: -p.production)
    
    # Always keep at least 1 planet to stay alive
    if len(my_sorted) <= TRIAGE_SAFE_PLANETS: return set()  
    
    for planet in my_sorted[TRIAGE_SAFE_PLANETS:]:
        if planet.production >= TRIAGE_MIN_PROD: continue  # never abandon high-prod
        
        tl = world.timelines[planet.id]
        if tl["holds_full"]: continue  # don't abandon safe planets
        
        keep = tl["keep_needed"]
        sv   = max(1, world.remaining - (tl["fall_turn"] or world.remaining))
        planet_val = planet.production * sv
        
        # Abandon if defense cost > 3x planet value
        if keep > planet_val * TRIAGE_COST_RATIO:
            abandon.add(planet.id)
            
    return abandon

# --- EXCERPT: VULNERABILITY WINDOW SCORING ---
# Stacks EXPOSED_VM (2.80x) with VULN_WINDOW_BONUS (2.20x) = 6.16x Multiplier!
if target.id in world.vuln_ids:
    val *= EXPOSED_VM
    if VULN_WINDOW_ENABLED: 
        val *= VULN_WINDOW_BONUS